## Import the package

We start by importing the `siibra` package and other necessary libraries.

In [2]:
pip install siibra

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 665.0/665.0 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 101.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.1/28.1 MB 65.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 85.7 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.5.0 which is incompatible.


In [ ]:
import siibra
import matplotlib
from nilearn import plotting

with siibra.QUIET:
    siibra.warm_cache()

## I. Find and display maps

### 1. Browse available parcellations

In [ ]:
siibra.parcellations.dataframe

In [ ]:
for parc in siibra.parcellations:
    print(parc.name)

### 2. Select Von Economo & Koskinas and display the region hierarchy

In [ ]:
voneconomo = siibra.parcellations["von economo"]
voneconomo.name

In [ ]:
voneconomo.render_tree()

In [ ]:
voneconomo.find("granularis")

In [ ]:
voneconomo.LICENSE

In [ ]:
voneconomo.publications

### 3. Get the labelled map of von economo koskinas

In [ ]:
voneconomo_map = voneconomo.get_map("MNI ICBM 2009c nonlinear asym")
img = voneconomo_map.fetch()
print(type(img))

### 4. Plot the map on MNI ICBM 2009c nonlinear asym

In [ ]:
plotting.view_img(
    img,
    cmap=voneconomo_map.get_colormap(),
    symmetric_cmap=False,
    resampling_interpolation="nearest"
)

### HANDS ON

Get the map of julich brain 3.1 and its map, then plot it.

In [ ]:
# Get julich brain 2.9 parcellation





In [ ]:
# Get the corresponding map in the reference space MNI ICBM 2009c nonlinear asym







In [ ]:
# Plot the map








## II. Using siibra-python to characterize motor and language areas in the Julich-Brain cytoarchitectonic atlas

### 1. Select precentral gyrus from Julich-Brain 3.1 parcellation

In [ ]:
julichbrain = siibra.parcellations.get("julich brain 3.1")
precentral_gyrus = julichbrain.get_region("precentral gyrus")
print("object type:", type(precentral_gyrus))
print("name:", precentral_gyrus)
print("parcellation:", precentral_gyrus.parcellation)

### HANDS ON: render tree of the "precentral gyrus" region

### 2. Plot the probability maps of these regions

In [ ]:
# fetch and display their probability maps
for region in precentral_gyrus.leaves:
    pmap = region.get_regional_map(
        space=siibra.spaces["MNI ICBM 2009c nonlinear asym"],
        maptype="statistical"
    )
    plotting.plot_stat_map(pmap.fetch(), title=region.name, cmap="magma")

### 6. Get receptor density profiles for area 4p

In [ ]:
area_4p = julichbrain.get_region("Area 4p (PreCG)")
print(area_4p)
receptor_profiles = siibra.features.get(area_4p, "receptor density profile")[0]
print(f"Found {len(receptor_profiles)} profiles are anchored at:", receptor_profiles.anchor)

In [ ]:
print(receptor_profiles.description)

In [ ]:
for rp in receptor_profiles:
    print(rp.receptor)

### 7. Plot `alpha4beta2` and `GABAB` receptor density distributions for both regions

In [ ]:
for receptor in ["alpha4beta2", "GABAB"]:
    receptor_profiles.get_element(receptor).plot(label=receptor)
matplotlib.pyplot.legend()

### HANDS ON: Plot `AMPA` receptor density distributions for both regions

In [ ]:
# Is AMPA the receptor we are looking for?





### 8. Run a differential gene expression analysis for the child regions of precentral gyrus

In [ ]:
# setup and run the analysis
genes = ['GABBR1', 'GABBR2']

geneexps = {}
for region in precentral_gyrus.children:
    geneexps[region] = siibra.features.get(region, "GeneExpression", gene=genes)[0]
    geneexps[region].plot()

In [ ]:
print(f"Microarray data for {region}")
geneexps[region].data

### 9. Plot the extracted sample positions of the microarray data

In [ ]:
area_4a = julichbrain.get_region("Area 4a (PreCG)")
geneexps[area_4a].data['mni_xyz']

In [ ]:
print(geneexps[area_4a].anchor.location.space)
print(geneexps[area_4a].anchor.location)

In [ ]:
for region in precentral_gyrus.children:
    region_mask = region.get_regional_mask("mni 152")
    display = plotting.plot_glass_brain(region_mask.fetch(), cmap="Blues", title=region.name, colorbar=False)
    display.add_markers(geneexps[region].anchor.location.coordinates, marker_size=5)

### 10. Plot the coordinates on BigBrain

In [ ]:
# get BigBrain space
bigbrain = siibra.spaces["bigbrain"]

In [ ]:
# warp the points to BigBrain space
points_in_mni152 = geneexps[area_4a].anchor.location
points_in_bigbrain = points_in_mni152.warp(bigbrain)
print(points_in_bigbrain.space)
print(points_in_bigbrain.coordinates)

### 11. Plot on BigBrain template

In [ ]:
bigbrain_template = bigbrain.get_template()
display = plotting.plot_img(bigbrain_template.fetch(resolution_mm=0.68), cmap="gray", cut_coords=points_in_bigbrain.centroid.coordinate)
display.add_markers(points_in_bigbrain.coordinates, marker_size=5)

## III. Download high-resolution BigBrain chunks

### HANDS ON
You can query for high resolution sections using the regions as well. In this case, can you query for 4p left?

In [ ]:
# No idea what the modality is? Try:
# siibra.features.render_ascii_tree("Feature")





### 1. While we could use the above query, we can also use an explorer link to obtain the volume of interest, aka bounding box

In [ ]:
from siibra import explorer

url = "https://atlases.ebrains.eu/viewer/#/a:juelich:iav:atlas:v1.0.0:1/t:minds:core:referencespace:v1.0.0:a1655b99-82f1-420f-a3c2-fe80fd4c8588/p:minds:core:parcellationatlas:v1.0.0:94c1125b-b87e-45e4-901c-00daee7f2579-bb/@:0.0.0.-W000.._eCwg.2-FUe3._-s_W.2_evlu..7LIy..BSR0.2_l20~.Qi-0..9I-/vs:v2-ff011b0b"
voi = explorer.decode_url(url).bounding_box
print(voi)

### 2.  Plot the BigBrain template for the view port of the above url

In [ ]:
chunk = bigbrain_template.fetch(voi=voi, resolution_mm=0.2)
plotting.view_img(chunk, bg_img=None, cmap='gray', vmin=0, vmax=255, symmetric_cmap=False)

### 3. Zoom in 10 percent and query for sections overlapping with this bounding box

In [ ]:
voi_zoomed = voi.zoom(0.05)
sections = siibra.features.get(voi_zoomed, "CellbodyStainedSection")
for section in sections:
    print(section.name)

### 4. Fetch and plot chunk of section 1049

In [ ]:
section = [s for s in sections if "1049" in s.name][0]
print(section)

In [ ]:
intersection = voi_zoomed.intersection(section)
section_chunk = section.fetch(voi=intersection)
fig = matplotlib.pyplot.figure(figsize=(10,10))
plotting.plot_img(section_chunk, bg_img=None, cmap='gray', display_mode='y', figure=fig, colorbar=False)